# 🔷 Day 01a: TensorFlow Basics — Tensors, Variables & GradientTape

---

## 🎯 What You'll Master Today
- `tf.constant`, `tf.Variable` — when to use which
- Tensor operations & NumPy interop
- `tf.GradientTape` — TF's autograd
- Eager execution vs Graph mode

**Goal:** Gradient descent from scratch using GradientTape.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  TensorFlow Core Concepts                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  tf.constant  → Immutable tensor (like torch.tensor)            ║
║  tf.Variable  → Mutable, trainable (like torch requires_grad)   ║
║                                                                   ║
║  GradientTape → Records operations for automatic differentiation║
║    with tf.GradientTape() as tape:                              ║
║        y = f(x)                                                  ║
║    grads = tape.gradient(y, [x])                                ║
║                                                                   ║
║  PyTorch vs TF:                                                  ║
║    torch: requires_grad=True, .backward()                       ║
║    tf:    tf.Variable + GradientTape                            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import tensorflow as tf
import numpy as np

# ============================================================
# 1. Tensor Creation
# ============================================================

# tf.constant — immutable
a = tf.constant([1, 2, 3], dtype=tf.float32)
b = tf.constant([[1, 2], [3, 4]], dtype=tf.float32)
print(f"Constant: {a}, shape={a.shape}, dtype={a.dtype}")

# Creation helpers
zeros = tf.zeros((2, 3))
ones = tf.ones((2, 3))
rng = tf.random.normal((2, 3), mean=0, stddev=1)
eye = tf.eye(3)
rng_uniform = tf.random.uniform((2, 3), minval=0, maxval=1)

print(f"Zeros: {zeros.shape}")
print(f"Random: {rng.shape}")

# tf.Variable — mutable, tracked for gradients
w = tf.Variable(tf.random.normal((3, 2)))
print(f"\nVariable: {w.shape}, trainable={w.trainable}")
w.assign(tf.ones((3, 2)))   # in-place update
w[0, 0].assign(99.0)        # element-wise assign
print(f"After assign: {w.numpy()}")

# NumPy ↔ TensorFlow
np_arr = np.array([1, 2, 3], dtype=np.float32)
tf_tensor = tf.constant(np_arr)    # numpy → tf
back_to_np = tf_tensor.numpy()     # tf → numpy
print(f"\nNumPy roundtrip: {back_to_np}")

In [ ]:
# ============================================================
# 2. Tensor Operations
# ============================================================

a = tf.constant([[1.0, 2], [3, 4]])
b = tf.constant([[5.0, 6], [7, 8]])

# Element-wise
print(f"Add: {(a + b).numpy()}")
print(f"Mul: {(a * b).numpy()}")

# Matrix multiplication
c = tf.matmul(a, b)     # or a @ b
print(f"MatMul: {c.numpy()}")

# Reshape, transpose
x = tf.constant([[1, 2, 3], [4, 5, 6]])
print(f"Reshape: {tf.reshape(x, (3, 2)).numpy()}")
print(f"Transpose: {tf.transpose(x).numpy()}")

# Reduction
print(f"Sum: {tf.reduce_sum(a).numpy()}")
print(f"Mean: {tf.reduce_mean(a).numpy()}")
print(f"Max: {tf.reduce_max(a).numpy()}")
print(f"Argmax: {tf.argmax(a, axis=1).numpy()}")

# Indexing/slicing works like numpy
print(f"Slice: {a[0, :].numpy()}")
print(f"Bool mask: {tf.boolean_mask(a, a > 2).numpy()}")

In [ ]:
# ============================================================
# 3. GradientTape — TF's Autograd
# ============================================================

# Basic gradient
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2  # y = x²
dy_dx = tape.gradient(y, x)
print(f"y = x², x=3 → dy/dx = {dy_dx.numpy()}")  # 2*3 = 6

# Multiple variables
w = tf.Variable(2.0)
b = tf.Variable(1.0)
with tf.GradientTape() as tape:
    y = w * 3.0 + b  # y = 3w + b
grads = tape.gradient(y, [w, b])
print(f"y = 3w + b → dw={grads[0].numpy()}, db={grads[1].numpy()}")

# IMPORTANT: GradientTape is one-shot by default!
x = tf.Variable(2.0)
with tf.GradientTape(persistent=True) as tape:
    y = x ** 3
    z = x ** 2
print(f"dy/dx = {tape.gradient(y, x).numpy()}")  # 3*4 = 12
print(f"dz/dx = {tape.gradient(z, x).numpy()}")  # 2*2 = 4
del tape  # must delete persistent tape manually

# Watching constants (not tracked by default)
x_const = tf.constant(3.0)
with tf.GradientTape() as tape:
    tape.watch(x_const)  # explicitly watch constants
    y = x_const ** 2
print(f"\nWatching constant: dy/dx = {tape.gradient(y, x_const).numpy()}")

In [ ]:
# ============================================================
# 4. Gradient Descent from Scratch with GradientTape
# ============================================================

# Generate synthetic data: y = 3x + 2 + noise
np.random.seed(42)
X_data = np.random.randn(100).astype(np.float32)
y_data = 3.0 * X_data + 2.0 + np.random.randn(100).astype(np.float32) * 0.1

# Learnable parameters
w = tf.Variable(0.0)
b = tf.Variable(0.0)
lr = 0.1

for epoch in range(100):
    with tf.GradientTape() as tape:
        y_pred = w * X_data + b
        loss = tf.reduce_mean((y_pred - y_data) ** 2)
    
    # Compute gradients
    dw, db = tape.gradient(loss, [w, b])
    
    # Update parameters (manual SGD)
    w.assign_sub(lr * dw)  # w -= lr * dw
    b.assign_sub(lr * db)
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.numpy():.4f}, w={w.numpy():.3f}, b={b.numpy():.3f}")

print(f"\nLearned: w={w.numpy():.3f} (true=3.0), b={b.numpy():.3f} (true=2.0)")
print("\n💡 GradientTape pattern: record → gradient → update → repeat")

In [ ]:
# ============================================================
# 5. Eager vs Graph Mode (@tf.function)
# ============================================================
import time

# Eager mode (default in TF2): runs like normal Python
def train_step_eager(x, y, w, b):
    with tf.GradientTape() as tape:
        pred = w * x + b
        loss = tf.reduce_mean((pred - y) ** 2)
    grads = tape.gradient(loss, [w, b])
    w.assign_sub(0.01 * grads[0])
    b.assign_sub(0.01 * grads[1])
    return loss

# Graph mode: compiled for speed
@tf.function
def train_step_graph(x, y, w, b):
    with tf.GradientTape() as tape:
        pred = w * x + b
        loss = tf.reduce_mean((pred - y) ** 2)
    grads = tape.gradient(loss, [w, b])
    w.assign_sub(0.01 * grads[0])
    b.assign_sub(0.01 * grads[1])
    return loss

x_t = tf.constant(X_data)
y_t = tf.constant(y_data)

# Benchmark eager
w1, b1 = tf.Variable(0.0), tf.Variable(0.0)
start = time.time()
for _ in range(1000):
    train_step_eager(x_t, y_t, w1, b1)
eager_time = time.time() - start

# Benchmark graph
w2, b2 = tf.Variable(0.0), tf.Variable(0.0)
train_step_graph(x_t, y_t, w2, b2)  # warmup / trace
start = time.time()
for _ in range(1000):
    train_step_graph(x_t, y_t, w2, b2)
graph_time = time.time() - start

print(f"Eager: {eager_time:.3f}s")
print(f"Graph: {graph_time:.3f}s")
print(f"Speedup: {eager_time/graph_time:.1f}×")
print("\n💡 @tf.function compiles Python into a TF graph")
print("   Use it for training loops in production")

---

## 🗺️ TF Basics Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  TENSORFLOW CORE                                                 ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  tf.constant  → immutable data                                  ║
║  tf.Variable  → trainable parameters                            ║
║                                                                   ║
║  GradientTape:                                                   ║
║    with tf.GradientTape() as tape:                              ║
║        loss = compute_loss(...)                                  ║
║    grads = tape.gradient(loss, [vars])                          ║
║    var.assign_sub(lr * grad)                                     ║
║                                                                   ║
║  Key differences from PyTorch:                                   ║
║    • No .backward() — use tape.gradient()                       ║
║    • No .zero_grad() — tape is per-use                          ║
║    • @tf.function for graph compilation                         ║
║    • Channels-last by default (B,H,W,C) not (B,C,H,W)         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | tf.constant vs tf.Variable? | constant=immutable data, Variable=mutable+trainable, tracked by GradientTape |
| 2 | How does GradientTape work? | Records ops in forward pass, then tape.gradient() computes grads. One-shot by default, use persistent=True for multi-grad |
| 3 | Eager vs Graph execution? | Eager=TF2 default, runs immediately like Python. Graph=@tf.function, compiled, faster but harder to debug |
| 4 | TF vs PyTorch key differences? | TF: channels-last, GradientTape, @tf.function, Keras API. PyTorch: channels-first, .backward(), define-by-run |
| 5 | What does @tf.function do? | Traces Python function into a TF graph. First call is slow (tracing), subsequent calls fast. Avoid Python side effects inside |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • tf.Variable for trainable params, tf.constant for data│
## │  • GradientTape records ops → tape.gradient() computes  │
## │  • var.assign_sub(lr * grad) for manual SGD              │
## │  • @tf.function for production-speed graph compilation  │
## │  • TF2 = eager by default (like PyTorch)                │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Keras Sequential** — high-level model building